<a href="https://colab.research.google.com/github/solivagvs/Stat-554/blob/main/sandorHW07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Name: Bryan Sandor

Course: Stat 554

Title: Homework 7

# Preamble

**Summary**: This homework assists in the practice of multiple linear regression (MLR) and logistic models. The data used is about wine quality subject to the following variables:
- Input variables
    1. fixed acidity
    2. volatile acidity
    3. citric acid
    4. residual sugar
    5. chlorides
    6. free sulfur dioxide
    7. total sulfur dioxide
    8. density
    9. pH
    10. sulphates
    11. alcohol
- Output variable
    1. quality (a score between $0$ and $10$)

Now, instead of using the data to predict the quality, the target for fitting MLR models is `alcohol` and for logistic regression models we use the `type` as the response.

# Read In and Combine Data

The data was obtained from the UCI machine learning repository website https://archive.ics.uci.edu/. It includes two sets of data, one for red wine and the other for white wine.

In [1]:
import pandas as pd
import numpy as np

redwine = pd.read_csv("winequality-red.csv", delimiter = ";")
whiwine = pd.read_csv("winequality-white.csv", delimiter = ";")

Now we combine the two datasets into a single data set, first preserving the type of wine in an additional _numeric_ column, $0$ for red and $1$ for white.

In [2]:
redwine["type"] = 0 # Assign red wines a code value of 0
whiwine["type"] = 1 # Assign white wines a code value of 1

In [3]:
winedat = pd.concat([redwine, whiwine], ignore_index = True)

Now we show a small random sample of the data to show the results are preserved.

In [4]:
winedat.sample(10, random_state = 1982) # random sample of 10 cells, with a seed

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
923,6.8,0.410,0.31,8.8,0.084,26.0,45.0,0.998240,3.38,0.64,10.1,6,0
5558,5.6,0.185,0.19,7.1,0.048,36.0,110.0,0.994380,3.26,0.41,9.5,6,1
1905,7.2,0.430,0.24,6.7,0.058,40.0,163.0,0.995000,3.20,0.41,9.9,5,1
1780,7.1,0.340,0.15,1.2,0.053,61.0,183.0,0.993600,3.09,0.43,9.2,5,1
1520,6.5,0.530,0.06,2.0,0.063,29.0,44.0,0.994890,3.38,0.83,10.3,6,0
6145,6.5,0.200,0.31,2.1,0.033,32.0,95.0,0.989435,2.96,0.61,12.0,6,1
1324,6.7,0.460,0.24,1.7,0.077,18.0,34.0,0.994800,3.39,0.60,10.6,6,0
2349,7.2,0.290,0.40,7.6,0.024,56.0,177.0,0.992800,3.04,0.32,11.5,6,1
4798,6.8,0.210,0.40,6.3,0.032,40.0,121.0,0.992140,3.18,0.53,12.0,7,1
4219,6.5,0.280,0.28,20.4,0.041,40.0,144.0,1.000200,3.14,0.38,8.7,5,1


# Split the Data

Now we will split up the data into a
1. training set and a
2. test set.

We must use stratified sampling to ensure we have similar proportions of red and white wines across the two sets.

In [5]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, LassoCV, Lasso

X_train, X_test, y_train, y_test = train_test_split(
  winedat.drop("alcohol", axis = 1),
  winedat["alcohol"],
  test_size = 0.20,
  stratify = winedat["type"],
  random_state = 1982)

Now we verify the initial proportion of red and whites across the data sets are preserved.

In [6]:
print("Original dataset class distribution:")
print(winedat["type"].value_counts(normalize=True))
print("\nTraining set class distribution:")
print(X_train.value_counts("type", normalize=True))
print("\nTesting set class distribution:")
print(X_test.value_counts("type", normalize=True))

Original dataset class distribution:
type
1    0.753886
0    0.246114
Name: proportion, dtype: float64

Training set class distribution:
type
1    0.753896
0    0.246104
Name: proportion, dtype: float64

Testing set class distribution:
type
1    0.753846
0    0.246154
Name: proportion, dtype: float64


# Regression Task

We will now train several models, using the variable `alcohol` as the response.

## Train Models

### Multiple Linear Regression (MLR) Models

#### Standardization

First we standardize all the variables so they all have mean $0$ and standard deviation $1$.

In [7]:
means = X_train.apply(np.mean, axis = 0)
means

,0
fixed acidity,7.223629
volatile acidity,0.340819
citric acid,0.319225
residual sugar,5.451924
chlorides,0.056014
free sulfur dioxide,30.438426
total sulfur dioxide,115.671349
density,0.994704
pH,3.216421
sulphates,0.530471


In [8]:
stds = X_train.apply(np.std, axis = 0)
stds

,0
fixed acidity,1.303257
volatile acidity,0.165255
citric acid,0.145772
residual sugar,4.791132
chlorides,0.034533
free sulfur dioxide,17.879413
total sulfur dioxide,56.664252
density,0.003014
pH,0.160284
sulphates,0.148753


The following divides each entry by its category's standard deviation after subtracting the category's mean from it, standardizing the entries.

In [9]:
X_train = X_train.apply(lambda x: (x - np.mean(x)) / np.std(x), axis = 0)
X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type
4235,-0.018131,-0.610081,-0.337682,1.241476,-0.637487,0.534781,1.029373,1.003997,-0.102450,-0.608198,-0.919828,0.571351
4011,-0.708708,0.237096,-1.023684,0.698807,0.694578,1.094084,1.699990,1.103529,1.582061,0.870761,-2.053955,0.571351
3617,-0.171592,-0.852132,0.142520,-0.699610,-0.203118,-1.031266,0.358756,-0.830710,0.209497,-0.473747,1.348426,0.571351
3783,-0.631977,-0.065467,-0.543483,0.907526,-0.434781,-0.080452,-0.117735,0.297319,0.459054,-0.406521,0.214299,0.571351
502,2.437255,0.600173,2.817930,0.229189,0.520830,0.422921,-0.700112,1.425349,-0.289617,2.148043,1.348426,-1.750237
...,...,...,...,...,...,...,...,...,...,...,...,...
4968,-0.171592,-0.731107,-0.269081,1.074501,-0.492697,0.087339,0.023448,-0.382816,-1.038289,0.131282,1.348426,0.571351
1007,1.439755,-0.247005,0.142520,-0.720482,0.231251,-1.031266,-1.600151,0.151339,0.271886,2.080817,1.348426,-1.750237
5244,-1.015631,-0.973158,-0.269081,-0.073453,-0.492697,1.094084,0.411700,-1.013186,-0.352007,-0.608198,0.214299,0.571351
4919,0.212062,-0.307518,0.279721,2.138968,-0.174160,-0.080452,0.146982,0.695447,-0.975899,0.064056,-0.919828,0.571351


We verify the standardization:

In [10]:
X_train.apply(np.mean, axis = 0)

,0
fixed acidity,-3.059148e-16
volatile acidity,8.203303e-18
citric acid,1.415070e-16
residual sugar,-1.189479e-16
chlorides,-7.519694e-18
free sulfur dioxide,-8.203303e-18
total sulfur dioxide,-2.529352e-17
density,2.134602e-14
pH,3.442995e-15
sulphates,-2.378958e-16


Numerically, the means are all zero.

In [11]:
X_train.apply(np.std, axis = 0)

,0
fixed acidity,1.0
volatile acidity,1.0
citric acid,1.0
residual sugar,1.0
chlorides,1.0
free sulfur dioxide,1.0
total sulfur dioxide,1.0
density,1.0
pH,1.0
sulphates,1.0


#### MLR Models

##### 1. Full Model

We begin with a "full" model, utilizing all variables as predictors:

In [12]:
cv_full_model = cross_validate(
    LinearRegression(),
    X_train,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

Then we find its root mean square error (RMSE).

In [13]:
print(np.sqrt(-sum(cv_full_model['test_score'])))

1.1649053114357433


##### 2. The Shelton Model

Privileged to work at a university, I am blessed to be friends with numerous colleagues across a variety of disciplines. One of my favorites, Dr Abigail Shelton, is an Inorganic Chemist. I asked her which variables, in her professional opinion, would best predict `alcohol` content and she said both `density` and `residual_sugar`. So I will try a model just utilizing those two variables.

In [14]:
cv_shelton_model = cross_validate(
    LinearRegression(),
    X_train[["residual sugar", "density"]],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

print(np.sqrt(-sum(cv_shelton_model['test_score'])))

1.9640109092873392


This RMSE is not much worse than the full model, and with only two variables it seems a "decent" trade-off.

##### 3. Shelton Model with Interaction

Let's redo this model, now including an additional interaction between `residual sugar` and `density`.

In [77]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree = 2, interaction_only = True)
X_poly = poly.fit_transform(X_train[["residual sugar", "density"]])

cv_shelton_int_model = cross_validate(
    LinearRegression(),
    X_poly,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

print(np.sqrt(-sum(cv_shelton_int_model['test_score'])))

2.0599272423652897


##### Shelton Model Polynomial

Finally, let's redo this model as a quadratic, also including the interaction between `residual sugar` and `density`.

In [96]:
from sklearn.preprocessing import PolynomialFeatures

poly2 = PolynomialFeatures(degree = 2, include_bias = False)
X_poly2 = poly2.fit_transform(X_train[["residual sugar", "density"]])

cv_shelton_poly_model = cross_validate(
    LinearRegression(),
    X_poly2,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

print(np.sqrt(-sum(cv_shelton_poly_model['test_score'])))

1.6550980931384518


##### Best MLR

In [109]:
mlr_best = LinearRegression().fit(X_train, y_train)
print(mlr_best.intercept_)
print(np.array(list(zip(X_train.columns, mlr_best.coef_))))

10.486192033865734
[['fixed acidity' '0.6695300391508433']
 ['volatile acidity' '0.13834093879468645']
 ['citric acid' '0.07871440904551724']
 ['residual sugar' '1.0874591552849']
 ['chlorides' '-0.03233277666806769']
 ['free sulfur dioxide' '-0.054173370041241176']
 ['total sulfur dioxide' '-0.018265414059569884']
 ['density' '-1.9529348734770697']
 ['pH' '0.4120223426516638']
 ['sulphates' '0.15013618832360404']
 ['quality' '0.09661297928684509']
 ['type' '-0.4766229213380797']]


### LASSO Model

Now we compute a LASSO model, using the two variables Dr Shelton suggested in addition to `citric acid`, `chlorides`, and `sulphates`.

In [17]:
lasso_model = LassoCV(cv = 5, random_state = 1982) \
    .fit(X_train[["residual sugar", "density",
                  "citric acid", "chlorides",
                  "sulphates"]],
         y_train)

In [18]:
np.set_printoptions(suppress = True)
fit_info = np.array(list(zip(lasso_model.alphas_, np.mean(lasso_model.mse_path_, axis = 1))))
fit_info[fit_info[:,1].argsort()][0:10,].round(4)

array([[0.0008, 0.7025],
       [0.0009, 0.7025],
       [0.0009, 0.7025],
       [0.001 , 0.7025],
       [0.0011, 0.7025],
       [0.0012, 0.7025],
       [0.0012, 0.7025],
       [0.0013, 0.7025],
       [0.0014, 0.7025],
       [0.0015, 0.7025]])

In [19]:
print(lasso_model.alpha_)
print(lasso_model.intercept_)
print(np.array(list(zip(X_train[["residual sugar", "density",
                  "citric acid", "chlorides",
                  "sulphates"]].columns,
                        lasso_model.coef_))))

0.0008118705476757677
10.486192033865713
[['residual sugar' '0.14336388757000065']
 ['density' '-0.9519316871324272']
 ['citric acid' '0.046672122880157554']
 ['chlorides' '-0.05220250603635444']
 ['sulphates' '0.2932671607931004']]


In [20]:
lasso_best = Lasso(lasso_model.alpha_).fit(X_train[["residual sugar", "density",
                  "citric acid", "chlorides",
                  "sulphates"]],
                                           y_train)

### Ridge Regression

In [43]:
from sklearn.linear_model import RidgeCV

ridge_model = RidgeCV(cv = None, store_cv_results = True) \
    .fit(X_train[["fixed acidity", "citric acid",
                  "chlorides", "pH", "sulphates"]],
         y_train)

In [44]:
print("Int:", ridge_model.intercept_, "Coeffs:", ridge_model.coef_)

Int: 10.486192033865693 Coeffs: [-0.0072419   0.04948523 -0.35468844  0.14765901  0.11447619]


In [47]:
print(ridge_model.alpha_)

10.0


In [48]:
print(ridge_model.best_score_)

-1.3024270376447766


In [49]:
print(ridge_model.cv_results_)

[[2.59773542 2.59767052 2.59702317]
 [1.27377425 1.27377364 1.27376721]
 [1.11727661 1.11728444 1.11736294]
 ...
 [0.5884837  0.58846315 0.5882586 ]
 [0.00055003 0.00055029 0.00055285]
 [5.17269735 5.1729691  5.17567763]]


In [50]:
from sklearn.linear_model import Ridge

ridge_best = Ridge(lasso_model.alpha_)\
             .fit(X_train[["fixed acidity", "citric acid",
                           "chlorides", "pH", "sulphates"]],
                                           y_train)

### Elastic Net Model

In [60]:
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import ElasticNet

elastic_model = ElasticNetCV(cv=5,
                    random_state = 1982,
                    l1_ratio = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1],
                    n_alphas = 50)

elastic_model.fit(X_train, y_train)

ElasticNetCV(cv=5,
             l1_ratio=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1],
             n_alphas=50, random_state=1982)

In [61]:
print(elastic_model.alpha_)

0.0021780017045612324


In [62]:
print(elastic_model.l1_ratio_)

1.0


In [64]:
en = ElasticNet(alpha = elastic_model.alpha_,
                l1_ratio = elastic_model.l1_ratio_)
en.fit(X_train, y_train)

ElasticNet(alpha=np.float64(0.0021780017045612324), l1_ratio=np.float64(1.0))

In [65]:
print(np.array(list(zip(X_train.columns, en.coef_))))

[['fixed acidity' '0.655356518545692']
 ['volatile acidity' '0.13601792408253496']
 ['citric acid' '0.07563295278676205']
 ['residual sugar' '1.0580270303431019']
 ['chlorides' '-0.03146481600687596']
 ['free sulfur dioxide' '-0.05076483375506273']
 ['total sulfur dioxide' '-0.023136444250897545']
 ['density' '-1.919407874112393']
 ['pH' '0.40228366440488766']
 ['sulphates' '0.1467370058255945']
 ['quality' '0.10047589225067885']
 ['type' '-0.4622039202015309']]


## Test Models

In [22]:
#quick function to standardize based off of a supplied mean and std
def my_std_fun(x, means, stds):
    return(x - means) / stds

#loop through the columns and use the function on each
for x in X_test.columns:
    X_test[x] = my_std_fun(X_test[x], means[x], stds[x])
X_test.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type
4508,-0.785439,-0.973158,-0.131881,-0.929201,-0.724361,-0.416033,-0.753056,-1.454444,0.084718,-0.070395,0.214299,0.571351
1192,-0.018131,-0.549569,0.348321,-0.616122,0.202293,-1.087196,-1.317786,-0.104127,1.894007,1.811916,1.348426,-1.750237
6234,-1.092362,0.025302,-1.160885,1.116245,-0.666445,-0.248242,0.076391,0.078349,0.209497,-0.272071,0.214299,0.571351
5915,-0.018131,1.477607,-0.817884,1.032757,0.868326,-0.751614,-0.382452,0.698765,-0.539174,-0.877099,-0.919828,0.571351
2204,-0.708708,-0.489056,1.171524,-0.824841,-0.116244,0.199200,0.323461,-0.764356,-1.412624,0.534634,0.214299,0.571351


In [110]:
mlr_pred = mlr_best.predict(X_test)

lasso_pred = lasso_best.predict(X_test[["residual sugar", "density",
                  "citric acid", "chlorides",
                  "sulphates"]])

ridge_pred = ridge_best.predict(X_test[["fixed acidity", "citric acid",
                           "chlorides", "pH", "sulphates"]])

en_pred = en.predict(X_test)

print(np.sqrt(mean_squared_error(y_test, mlr_pred)), np.sqrt(mean_squared_error(y_test, lasso_pred)), np.sqrt(mean_squared_error(y_test, ridge_pred)), np.sqrt(mean_squared_error(y_test, en_pred)))

0.43935137478940806 0.7934887183190167 1.1240592166805483 0.44063090132590027


In [111]:
from sklearn.metrics import mean_absolute_error

print(mean_absolute_error(y_test, mlr_pred), mean_absolute_error(y_test, lasso_pred), mean_absolute_error(y_test, ridge_pred), mean_absolute_error(y_test, en_pred))

0.334105575355719 0.6347384719944269 0.9325585892097202 0.33564029776000254


# Classification Task